In [31]:
from itertools import product
from PIL import Image
import numpy as np
import colorsys

In [32]:
delta = {
    ('0', 'A'): ('1', 'B', 'R'),
    ('0', 'B'): ('1', 'C', 'R'),
    ('0', 'C'): ('1', 'D', 'R'),
    ('0', 'D'): ('1', 'A', 'L'),
    ('0', 'E'): ('1', 'Z', 'R'),
    ('1', 'A'): ('1', 'C', 'L'),
    ('1', 'B'): ('1', 'B', 'R'),
    ('1', 'C'): ('0', 'E', 'L'),
    ('1', 'D'): ('1', 'D', 'L'),
    ('1', 'E'): ('0', 'A', 'L')
}
gamma = ['0','1']
sigma = ['0']
Q = ['A','B','C','D','E','F', 'Z']
b = '0'
q0 = 'A'
F = ['Z']


In [33]:
n=2
gamma_primed = list(product(gamma, Q+['k']))
sigma_primed = list(product(sigma, [q0, 'k']))
b_primed = (b, 'k')
input_tuples = list(product(gamma_primed, repeat=2))

F_primed = [((x1,q1),(x2,q2)) for ((x1,q1),(x2,q2)) in input_tuples if q1 in F and q2 == 'k'] + [((x1,q1),(x2,q2)) for ((x1,q1),(x2,q2)) in input_tuples if q2 in F and q1 == 'k']

def construct_delta_prime():
    """
    Construct delta_prime based on the given transition rules.
    
    Args:
        delta: Dictionary representing original transition function
        gamma_primed: List of elements in Gamma'
    
    Returns:
        Dictionary representing delta_prime transition function
    """
    delta_prime = {}
    
    # Get all possible elements from gamma_primed
    gamma_prime_elements = gamma_primed
    
    # Find elements ending with 'k' (κ)
    elements_with_k = [(x, q) for x, q in gamma_prime_elements if q == 'k']
    
    # Find elements not ending with 'k'
    elements_without_k = [(x, q) for x, q in gamma_prime_elements if q != 'k']
    
    # Rule 1: δ′((x1, q1), (x2, κ)) = (((xout, qout), (x2, κ)), L) if δ(x1, q1) = (xout, qout, L)
    for elem1 in elements_without_k:
        x1, q1 = elem1
        for elem2 in elements_with_k:
            x2, k = elem2
            if (x1, q1) in delta:
                xout, qout, direction = delta[(x1, q1)]
                if direction == 'L':
                    delta_prime[((x1, q1), (x2, k))] = (((xout, qout), (x2, k)), 'L')
    
    # Rule 2: δ′((x1, κ), (x2, q2)) = (((xout, qout), (x2, κ)), L) if δ(x1, q2) = (xout, qout, L)
    for elem1 in elements_with_k:
        x1, k = elem1
        for elem2 in elements_without_k:
            x2, q2 = elem2
            if (x1, q2) in delta:
                xout, qout, direction = delta[(x1, q2)]
                if direction == 'L':
                    delta_prime[((x1, k), (x2, q2))] = (((xout, qout), (x2, k)), 'L')
    
    # Rule 3: δ′((x1, q1), (x2, κ)) = (((xout, κ), (x2, qout)), R) if δ(x1, q1) = (xout, qout, R)
    for elem1 in elements_without_k:
        x1, q1 = elem1
        for elem2 in elements_with_k:
            x2, k = elem2
            if (x1, q1) in delta:
                xout, qout, direction = delta[(x1, q1)]
                if direction == 'R':
                    delta_prime[((x1, q1), (x2, k))] = (((xout, k), (x2, qout)), 'R')
    
    # Rule 4: δ′((x1, κ), (x2, q2)) = (((xout, κ), (x2, qout)), R) if δ(x1, q2) = (xout, qout, R)
    for elem1 in elements_with_k:
        x1, k = elem1
        for elem2 in elements_without_k:
            x2, q2 = elem2
            if (x1, q2) in delta:
                xout, qout, direction = delta[(x1, q2)]
                if direction == 'R':
                    delta_prime[((x1, k), (x2, q2))] = (((xout, k), (x2, qout)), 'R')
    
    return delta_prime

delta_prime = construct_delta_prime()

In [34]:
def ma_step(tape,head,delta):
    input_slice = tuple(tape[head:head+2])
    (output_slice, dir) = delta[input_slice]
    tape[head:head+2] = list(output_slice)

    if dir == 'R':
        head +=1
    else:
        head -=1
    return (tape,head)

code_dict = {}
for (i,x) in enumerate(gamma_primed):
    code_dict[x] = i

def tape_to_code(tape):
    out_tape = []    
    for x in tape:
        out_tape.append(code_dict[x])
    return out_tape

def print_tape(tape):
    for x in tape:
        print("{:02d}".format(x), end=' ')
    print()

def print_tape(tape):
    # ANSI color codes for different ranges of numbers
    def get_color(num):
        if 0 <= num <= 9:
            colors = {
                0: '\033[37m',  # White
                1: '\033[32m',  # Green
                2: '\033[33m',  # Yellow
                3: '\033[34m',  # Blue
                4: '\033[35m',  # Purple
                5: '\033[36m',  # Cyan
                6: '\033[31m',  # Red
                7: '\033[33;1m',  # Bright yellow
                8: '\033[34;1m',  # Bright blue
                9: '\033[35;1m',  # Bright purple
            }
            return colors.get(num, '\033[37m')  # Default to white
        elif 10 <= num <= 19:
            return '\033[32m'    # Green
        elif 20 <= num <= 29:
            return '\033[33m'    # Yellow
        elif 30 <= num <= 39:
            return '\033[34m'    # Blue
        elif 40 <= num <= 49:
            return '\033[35m'    # Purple
        elif 50 <= num <= 59:
            return '\033[36m'    # Cyan
        elif 60 <= num <= 69:
            return '\033[31m'    # Red
        elif 70 <= num <= 79:
            return '\033[33;1m'  # Bright yellow
        elif 80 <= num <= 89:
            return '\033[34;1m'  # Bright blue
        elif 90 <= num <= 100:
            return '\033[35;1m'  # Bright purple
        else:
            return '\033[37m'    # Default to white
    
    reset = '\033[0m'  # Reset to default color
    
    for x in tape:
        color = get_color(x)
        # Format with leading zeros if needed
        if x < 10:
            print(f"{color}0{x}{reset}", end=' ')
        else:
            print(f"{color}{x}{reset}", end=' ')
    print()

In [40]:
N = 100
tape = [('0','k')] * N
head = N // 2
tape[head] = ('0',q0)

tape_lst = []
head_lst = []
# Run until an error occurs
i = 0
while True:
    try:
        tape, head = ma_step(tape, head, delta_prime)
        tape_lst.append(tape.copy())
        head_lst.append(head)
        i += 1  # Increment step count
    except Exception as e:
        print(f"Error occurred at step {i}: {e}")
        break  # Exit the loop if an error occurs


# Format the tapes for printing and print each one
fmt_tapes = [tape_to_code(tape) for tape in tape_lst]
#for tape in fmt_tapes:
    #print_tape(tape)

Error occurred at step 1271: ()


In [41]:
import numpy as np
from PIL import Image

def generate_image_from_array(arrs):
    # Define a function to map numbers to colors using HSL
    def get_hsl_color(num):
        # Map num to a hue value in the range [0, 1] for a rainbow of colors
        num = num * 4
        hue = (num % 360) / 360.0  # Ensures the hue is between 0 and 1
        saturation = 0.8  # Keep saturation high for bright colors
        lightness = 0.5  # Keep lightness in the middle for vibrant colors
        
        # Convert the HSL color to RGB
        r, g, b = colorsys.hls_to_rgb(hue, lightness, saturation)
        
        # Convert to 0-255 range
        r = int(r * 255)
        g = int(g * 255)
        b = int(b * 255)
        
        return (r, g, b)
    
    # Get the dimensions of the image
    height = len(arrs)  # Number of rows
    width = len(arrs[0]) if height > 0 else 0  # Number of columns (length of the first list)

    # Create an empty NumPy array to hold the pixel data
    img_array = np.zeros((height, width, 3), dtype=np.uint8)

    # Iterate through the list of lists and set pixel values based on the number
    for y in range(height):
        for x in range(width):
            color = get_hsl_color(arrs[y][x])
            # Set the pixel value at (x, y) to the color generated from the number
            img_array[y, x] = color

    # Convert the NumPy array into a Pillow image and save it
    img = Image.fromarray(img_array)
    img.save('generated_image.png')
    img.show()

generate_image_from_array(fmt_tapes)
